In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/summary/summary2.json


In [3]:
pip install -U langchain-community`

/bin/bash: -c: line 1: unexpected EOF while looking for matching ``'
/bin/bash: -c: line 2: syntax error: unexpected end of file
Note: you may need to restart the kernel to use updated packages.


In [8]:
!pip install protobuf==3.20.3

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 4.2 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tens

In [9]:
import json
import random
import numpy as np
import os
import torch
from collections import defaultdict

# --- CONFIGURATION ---
INPUT_FILE = "/kaggle/input/summary/summary2.json"
OUTPUT_FILE = "clusters_final_local.json"
# This model will be downloaded once and run LOCALLY on your hardware
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2" 

# --- ROBUST IMPORT HANDLING ---
# We define a fallback class that uses sentence-transformers directly
# This ensures the code runs even if langchain/langchain-community is broken or missing.
class NativeHuggingFaceEmbeddings:
    def __init__(self, model_name, model_kwargs=None, encode_kwargs=None):
        from sentence_transformers import SentenceTransformer
        device = model_kwargs.get("device", "cpu") if model_kwargs else "cpu"
        print(f"⚠ Using Native Fallback: Loading {model_name} on {device}...")
        self.model = SentenceTransformer(model_name, device=device)
        self.encode_kwargs = encode_kwargs or {}

    def embed_documents(self, texts):
        # Returns a list of vectors (numpy arrays converted to list)
        return self.model.encode(texts, **self.encode_kwargs).tolist()

    def embed_query(self, text):
        return self.model.encode(text, **self.encode_kwargs).tolist()

# Try loading LangChain wrappers, fallback to Native if they fail
HuggingFaceEmbeddings = None

try:
    # 1. Newest Version
    from langchain_huggingface import HuggingFaceEmbeddings
except (ImportError, ModuleNotFoundError):
    try:
        # 2. Older Version
        from langchain_community.embeddings import HuggingFaceEmbeddings
    except (ImportError, ModuleNotFoundError):
        try:
            # 3. Legacy Version
            from langchain.embeddings import HuggingFaceEmbeddings
        except (ImportError, ModuleNotFoundError):
            pass

# If all LangChain imports failed, use the Native Fallback
if HuggingFaceEmbeddings is None:
    print("⚠ LangChain modules not found. Using native sentence-transformers fallback.")
    HuggingFaceEmbeddings = NativeHuggingFaceEmbeddings

# --- 1. ROBUST DATA LOADER ---
def load_data_robust(filepath):
    print(f"Loading {filepath}...")
    if not os.path.exists(filepath):
        print(f"❌ File not found: {filepath}")
        return []

    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        content = f.read().strip()

    # Attempt 1: Standard JSON
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        print("⚠ JSON corrupted. Attempting repair...")

    # Attempt 2: Backtracking Repair (Fixes truncated files)
    try:
        # Find the last valid object separator "},"
        last_comma_idx = content.rfind('},')
        for _ in range(15): # Try 15 times to find a clean break
            if last_comma_idx == -1: break
            
            # Reconstruct valid JSON array
            candidate = content[:last_comma_idx+1] + "]"
            try:
                data = json.loads(candidate)
                print(f"✔ Recovered {len(data)} valid records.")
                return data
            except json.JSONDecodeError:
                # Move back to the previous object
                last_comma_idx = content.rfind('},', 0, last_comma_idx)
    except:
        pass
        
    print("❌ Failed to salvage data.")
    return []

# --- 2. CLUSTERING UTILS ---
def cluster_docs(total_docs):
    """Groups documents by Domain -> Intent"""
    clustered = defaultdict(lambda: defaultdict(list))
    for doc in total_docs:
        domain = doc.get("domain", "Unknown")
        intent = doc.get("intent", "Unknown")
        clustered[domain][intent].append(doc)
    return {d: dict(intents) for d, intents in clustered.items()}

def cosine_distance(a, b):
    # Manual Cosine Distance Calculation
    return 1 - np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# --- 3. MAIN EXECUTION ---
def run_pipeline():
    # 1. Load Data
    data = load_data_robust(INPUT_FILE)
    if not data: return

    # 2. Initialize Model LOCALLY
    # check for GPU
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\nLoading Model '{MODEL_NAME}' on {device.upper()}...")
    
    # Initialize the wrapper (either LangChain or our Native Fallback)
    emb_model = HuggingFaceEmbeddings(
        model_name=MODEL_NAME,
        model_kwargs={'device': device},
        encode_kwargs={'normalize_embeddings': True} # Normalization helps cosine distance accuracy
    )

    # 3. Process Clusters
    dicc = cluster_docs(data)
    results = {}

    for domain, intents in dicc.items():
        print(f"Processing Domain: {domain}")
        domain_runs = []
        intent_keys = list(intents.keys())
        
        # Try to generate 15 selections per domain
        for _ in range(15):
            if not intent_keys: break
            
            chosen_intent = random.choice(intent_keys)
            docs = intents[chosen_intent]

            if len(docs) < 3: continue

            # Extract summaries
            summaries = [d.get("summary", "") for d in docs]

            # Generate Embeddings (Locally)
            # This happens on your GPU/CPU, no API call
            embeddings = np.array(emb_model.embed_documents(summaries))

            # --- Anchor Selection Loop ---
            for __ in range(10):
                # A. Random Anchor
                anchor_idx = random.randint(0, len(summaries) - 1)
                anchor_doc = docs[anchor_idx]
                anchor_emb = embeddings[anchor_idx]

                # B. Compute Distances
                distances = []
                for i, emb in enumerate(embeddings):
                    if i == anchor_idx: continue
                    dist = cosine_distance(anchor_emb, emb)
                    distances.append((dist, docs[i])) # Store full object

                # C. Sort Descending (Farthest first)
                distances.sort(reverse=True, key=lambda x: x[0])

                # D. Select Top 10 Farthest + Anchor
                farthest_docs = [d for _, d in distances[:10]]
                farthest_docs.append(anchor_doc)

                # E. Clean Metadata Preservation
                # We create a new dict to ensure we keep exactly what we want
                cleaned_selection = []
                for doc in farthest_docs:
                    cleaned_selection.append({
                        "transcript_id": doc.get("transcript_id", "N/A"),
                        "domain": doc.get("domain", domain),
                        "intent": doc.get("intent", chosen_intent),
                        "summary": doc.get("summary", ""),
                        "reason_for_call": doc.get("reason_for_call", "N/A")
                    })

                domain_runs.append({
                    "intent": chosen_intent,
                    "selected_transcripts": cleaned_selection 
                })

        results[domain] = domain_runs

    # 4. Save Final Output
    # Formatting as { Domain: [Runs] }
    final_output = {domain: runs for domain, runs in results.items()}

    print(f"\nSaving to {OUTPUT_FILE}...")
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(final_output, f, indent=4, ensure_ascii=False)
    
    print("Done! Local processing complete.")

if __name__ == "__main__":
    run_pipeline()

⚠ LangChain modules not found. Using native sentence-transformers fallback.
Loading /kaggle/input/summary/summary2.json...

Loading Model 'sentence-transformers/all-MiniLM-L6-v2' on CPU...
⚠ Using Native Fallback: Loading sentence-transformers/all-MiniLM-L6-v2 on cpu...
Processing Domain: Hotel
Processing Domain: Flight
Processing Domain: Retail
Processing Domain: Banking
Processing Domain: Telecom
Processing Domain: Insurance

Saving to clusters_final_local.json...
Done! Local processing complete.
